In [1]:
# ATTENTION: only run this cell when on google colab
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Change the current directory to the "Drosophila" folder, which you might need to make.
%cd /content/drive/MyDrive/Drosophila

Mounted at /content/drive
/content/drive/MyDrive/Drosophila


In [2]:
!git clone https://github.com/philshiu/Drosophila_brain_model.git
!pip install brian2
%cd Drosophila_brain_model

Cloning into 'Drosophila_brain_model'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 183 (delta 45), reused 43 (delta 43), pack-reused 125 (from 2)
Receiving objects: 100% (183/183), 185.30 MiB | 15.02 MiB/s, done.
Resolving deltas: 100% (77/77), done.
Updating files: 100% (19/19), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.2 MB/s eta 0:00:00
/content/drive/MyDrive/Drosophila/Drosophila_brain_model


In [3]:
from model import run_exp
from model import default_params as params
import utils as utl
from brian2 import Hz

config = {
    'path_res'  : './results',                 # directory to store results, you may need to create this
    'path_comp' : './Completeness_783.csv',        # csv of the complete list of Flywire neurons, 783.
    'path_con'  : './Connectivity_783.parquet',    # connectivity data, for 783
    'n_proc'    : -1,                              # number of CPU cores (-1: use all)
}

List all of our output neurons. These are the neuron IDs of the descending neurons that we care about. These descending neurons may be used to control the body of the fly.

In [4]:
flyid2name = { }
#P9_oDN1 corresponds to forward velocity
P9_oDN1_left = 720575940626730883
P9_oDN1_right = 720575940620300308
P9_left = 720575940627652358
P9_right =720575940635872101
#DNa01 and DNa02 correspond to turning
DNa01_right = 720575940627787609
DNa01_left = 720575940644438551
DNa02_right = 720575940629327659
DNa02_left = 720575940604737708
#MDN, "Moonwalker descending neurons" are backwards walking/escape/startle response
MDN_1, MDN_2, MDN_3, MDN_4 = 720575940616026939, 720575940631082808,720575940640331472,720575940610236514
#Giant Fiber corresponds to escape
Giant_Fiber_1, Giant_Fiber_2 = 720575940622838154,720575940632499757
#MN9 corresponds to proboscis motor neuron, and corresponds to feeding.
MN9_left = 720575940660219265
MN9_right = 720575940618238523
#aDN1 correspond to antennal grooming
aDN1_right = 720575940616185531
aDN1_left = 720575940624319124


flyid2name[P9_oDN1_left]=       "P9_oDN1_left"
flyid2name[P9_oDN1_right]=      "P9_oDN1_right"
flyid2name[P9_left]=    "P9_left"
flyid2name[P9_right]=   "P9_right"
flyid2name[DNa01_right]=        "DNa01_right"
flyid2name[DNa01_left]= "DNa01_left"
flyid2name[DNa02_right]=        "DNa02_right"
flyid2name[DNa02_left]= "DNa02_left"
flyid2name[MDN_1]= "MDN_1"
flyid2name[MDN_2]= "MDN_2"
flyid2name[MDN_3]= "MDN_3"
flyid2name[MDN_4]= "MDN_4"
flyid2name[Giant_Fiber_1]= "Giant_Fiber_1"
flyid2name[Giant_Fiber_2]= "Giant_Fiber_2"
flyid2name[MN9_left]= "MN9_left"
flyid2name[MN9_right]= "MN9_right"
flyid2name[aDN1_right]= "aDN1_right"
flyid2name[aDN1_left]= "aDN1_left"

In [5]:
output_neurons = [P9_oDN1_left, P9_oDN1_right, DNa01_right, DNa01_left, DNa02_right, DNa02_left, MDN_1, MDN_2, MDN_3, MDN_4, Giant_Fiber_1, Giant_Fiber_2, MN9_left, MN9_right, aDN1_right, aDN1_left]

# Activate labellar sugar GRNs.

In [6]:
sugar_GRNs = [720575940616885538,720575940630233916,720575940639332736,720575940632889389,720575940617000768,720575940632425919,720575940637568838,720575940629176663,720575940621502051,720575940638202345,720575940612670570,720575940611875570,720575940621754367,720575940633143833,720575940613601698,720575940630797113,720575940639198653,720575940639259967,720575940624963786,720575940640649691,720575940610788069,720575940623172843,720575940628853239]

In [7]:
params['r_poi'] = 200 * Hz
run_exp(exp_name='Sugar_200Hz', neu_exc=sugar_GRNs, params=params, **config)

>>> Experiment:     Sugar_200Hz
    Output file:    results/Sugar_200Hz.parquet
    Excited neurons: 23


WARNING    /usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]
  warnings.warn(



    Elapsed time:   1101 s


In [8]:
ps = [
    './results/Sugar_200Hz.parquet',
]
df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
df_rate


exp_name,name,Sugar_200Hz
flyid,,
720575940605420902,,1.233333
720575940605513649,,4.066667
720575940605658033,,25.533333
720575940605682790,,18.400000
720575940605890784,,6.900000
...,...,...
720575940660219265,MN9_left,91.666667
720575940660220289,,4.133333
720575940660223873,,62.600000


In [9]:
relevant_neurons = [neuron for neuron in output_neurons if neuron in df_rate.index]
display(df_rate.loc[relevant_neurons])

exp_name,name,Sugar_200Hz
flyid,,
720575940660219265,MN9_left,91.666667
720575940618238523,MN9_right,62.200000


#Add bitter neurons

In [10]:
bitter_GRNs = [720575940619072513,720575940646212996,720575940622298631,720575940642088333,720575940627692048,720575940617239197,720575940618682526,
               720575940604714528,720575940603266592,720575940604027168,720575940619197093,720575940610259370,720575940627578156,720575940629481516,
               720575940618887217,720575940614281266,720575940634859188,720575940645743412,720575940637742911,720575940617094208,720575940629416318,
               720575940630195909,720575940615641798,720575940638312262,720575940624310345,720575940621778381,720575940619659861,720575940629146711,
               720575940625750105,720575940610483162,720575940610481370,720575940602353632,720575940610773090,720575940617433830,720575940628962407,
               720575940626287336,720575940623183083,720575940618025199,720575940619028208,720575940621864060,720575940613061118,720575940621008895,
               ]

In [11]:
for i, neuron_id in enumerate(bitter_GRNs):
    flyid2name[neuron_id] = f"bitter_GRN_{i+1}"

In [12]:
Activation_freq=200
params['r_poi'] = 200 * Hz
params['r_poi2'] = Activation_freq * Hz
run_exp(exp_name='Sugar_and_bitter', neu_exc=sugar_GRNs, neu_exc2=bitter_GRNs, params=params, **config)

>>> Experiment:     Sugar_and_bitter
    Output file:    results/Sugar_and_bitter.parquet
    Excited neurons: 65


WARNING    /usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]
  warnings.warn(



    Elapsed time:   2141 s


In [13]:
ps = [
    './results/Sugar_200Hz.parquet',
    './results/Sugar_and_bitter.parquet',
]
df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
df_rate


exp_name,name,Sugar_200Hz,Sugar_and_bitter
flyid,,,
720575940602353632,bitter_GRN_32,NaN,192.366667
720575940603266592,bitter_GRN_9,NaN,195.966667
720575940604027168,bitter_GRN_10,NaN,198.100000
720575940604714528,bitter_GRN_8,NaN,196.066667
720575940604934502,,NaN,1.066667
...,...,...,...
720575940660220289,,4.133333,NaN
720575940660223873,,62.600000,2.200000
720575940660224641,,2.800000,0.033333


In [14]:
relevant_neurons = [neuron for neuron in output_neurons if neuron in df_rate.index]
display(df_rate.loc[relevant_neurons])

exp_name,name,Sugar_200Hz,Sugar_and_bitter
flyid,,,
720575940660219265,MN9_left,91.666667,0.633333
720575940618238523,MN9_right,62.200000,0.033333


#First, run P9s to simulate just forward walking.

In [15]:
# run with different frequency
P9s = [P9_left, P9_right]
params['r_poi'] = 100 * Hz
run_exp(exp_name='P9s_100Hz', neu_exc=P9s, params=params, **config)

>>> Experiment:     P9s_100Hz
    Output file:    results/P9s_100Hz.parquet
    Excited neurons: 2


WARNING    /usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]
  warnings.warn(



    Elapsed time:   586 s


In [16]:
ps = [    './results/P9s_100Hz.parquet',]
df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
df_rate.sort_values('P9s_100Hz', ascending=False, inplace=True)
df_rate


exp_name,name,P9s_100Hz
flyid,,
720575940635872101,P9_right,99.800000
720575940627652358,P9_left,96.600000
720575940613626521,,37.033333
720575940633901395,,32.366667
720575940631924584,,26.700000
...,...,...
720575940616471052,,0.033333
720575940628881576,,0.033333
720575940624967463,,0.033333


In [17]:
relevant_neurons = [neuron for neuron in output_neurons if neuron in df_rate.index]
display(df_rate.loc[relevant_neurons])

exp_name,name,P9s_100Hz
flyid,,
720575940626730883,P9_oDN1_left,14.866667
720575940620300308,P9_oDN1_right,11.900000
720575940629327659,DNa02_right,6.466667
720575940604737708,DNa02_left,17.466667


#Vision: Looming → Escape

In [18]:
from IPython.display import Image
Image(url='https://ars.els-cdn.com/content/image/1-s2.0-S0960982219301381-fx1_lrg.jpg', width=800)

In [19]:
LC_4s =[720575940605598892,720575940611134833,720575940612580977,720575940613256863,720575940613260959,720575940614914107,720575940615462587,720575940617176321,720575940617266722,720575940618807105,720575940620795728,720575940622108001,720575940624017251,720575940625038090,720575940625934973,720575940625991043,720575940626605200,720575940626626895,720575940628454522,720575940628462340,720575940630851036,720575940638496720,720575940603637438,720575940610522009,720575940612093351,720575940612323025,720575940612380723,720575940612498129,720575940612518055,720575940612968421,720575940613609484,720575940613638041,720575940614572742,720575940614582946,720575940615053580,720575940615127227,720575940615232217,720575940615575007,720575940616066705,720575940616713355,720575940617026260,720575940617348379,720575940618002644,720575940618234704,720575940618234715,720575940618266459,720575940618267227,720575940618275520,720575940618312606,720575940618676440,720575940618709158,720575940618723749,720575940619397542,720575940620314221,720575940620314612,720575940620731380,720575940620903551,720575940621145821,720575940621522458,720575940621753579,720575940622330582,720575940622531767,720575940622939836,720575940624111763,720575940624790781,720575940624856762,720575940625841351,720575940625845447,720575940625906702,720575940625932421,720575940626553596,720575940626916936,720575940627519107,720575940628064260,720575940628081541,720575940628419527,720575940628518400,720575940628599895,720575940628606713,720575940628699560,720575940628891863,720575940629753807,720575940629964591,720575940630154660,720575940630484495,720575940630998339,720575940631032657,720575940631338271,720575940632475449,720575940632715234,720575940632769180,720575940633013355,720575940633218863,720575940633580384,720575940634517856,720575940635835967,720575940636957006,720575940638456227,720575940639817947,720575940640612480,720575940641213824,720575940645821316,720575940649229433,720575940652611745]

In [20]:
Activation_freq=200
params['r_poi'] = 100 * Hz
params['r_poi2'] = Activation_freq * Hz
run_exp(exp_name='P9_LC4s', neu_exc=P9s, neu_exc2=LC_4s, params=params, **config)

>>> Experiment:     P9_LC4s
    Output file:    results/P9_LC4s.parquet
    Excited neurons: 106


WARNING    /usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]
  warnings.warn(



    Elapsed time:   3560 s


In [21]:
ps = [
    './results/P9s_100Hz.parquet',
    './results/P9_LC4s.parquet',
]

df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
df_rate.sort_values('P9s_100Hz', ascending=False, inplace=True)
df_rate


exp_name,name,P9_LC4s,P9s_100Hz
flyid,,,
720575940635872101,P9_right,99.600000,99.800000
720575940627652358,P9_left,97.866667,96.600000
720575940613626521,,22.966667,37.033333
720575940633901395,,35.033333,32.366667
720575940631924584,,22.333333,26.700000
...,...,...,...
720575940653310198,,0.866667,NaN
720575940653399030,,7.366667,NaN
720575940655000737,,5.400000,NaN


In [22]:
relevant_neurons = [neuron for neuron in output_neurons if neuron in df_rate.index]
display(df_rate.loc[relevant_neurons])

exp_name,name,P9_LC4s,P9s_100Hz
flyid,,,
720575940626730883,P9_oDN1_left,0.333333,14.866667
720575940620300308,P9_oDN1_right,0.300000,11.900000
720575940627787609,DNa01_right,10.000000,NaN
720575940644438551,DNa01_left,9.600000,NaN
720575940629327659,DNa02_right,2.033333,6.466667
720575940604737708,DNa02_left,0.200000,17.466667
720575940616026939,MDN_1,7.533333,NaN
720575940631082808,MDN_2,21.233333,NaN
720575940640331472,MDN_3,16.933333,NaN


# Olfaction: Aversive → Repulsion

In [23]:
Image(url='https://www.cell.com/cms/10.1016/j.cell.2012.09.046/asset/c52d2646-b618-4a0d-bf07-4fe6c9d622fb/main.assets/fx1_lrg.jpg', width=800)

In [24]:
Or56a=[720575940659222657,720575940641403021,720575940624211470,720575940616536209,720575940615427734,720575940628380827,720575940654069409,720575940613671330,720575940644590116,720575940612972328,720575940627318696,720575940627805096,720575940632190765,720575940633031085,720575940634955188,720575940621106102,720575940615923131,720575940608928324,720575940631467591,720575940622553420,720575940628086607,720575940626357586,720575940632041043,720575940618946901,720575940616095318,720575940626411097,720575940634614367,720575940603832288,720575940620055905,720575940609633378,720575940637704676,720575940638202852,720575940622713578,720575940635705963,720575940629830508,720575940630257772,720575940619539182,720575940612019442,720575940639931893]

In [ ]:
Or56a_freq=250
params['r_poi'] = 100 * Hz
params['r_poi2'] = Or56a_freq * Hz
run_exp(exp_name='P9_Or56a', neu_exc=P9s, neu_exc2=Or56a, params=params, **config)

>>> Experiment:     P9_Or56a
    Output file:    results/P9_Or56a.parquet
    Excited neurons: 41


WARNING    /usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]
  warnings.warn(



In [ ]:
ps = [
    './results/P9s_100Hz.parquet',
    './results/P9_Or56a.parquet'
]

df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
df_rate.sort_values('P9s_100Hz', ascending=False, inplace=True)



In [29]:
relevant_neurons = [neuron for neuron in output_neurons if neuron in df_rate.index]
display(df_rate.loc[relevant_neurons].fillna(0))

exp_name,name,P9_Or56a,P9s_100Hz
flyid,,,
720575940626730883,P9_oDN1_left,0.366667,14.866667
720575940620300308,P9_oDN1_right,1.233333,11.900000
720575940627787609,DNa01_right,17.966667,0.000000
720575940644438551,DNa01_left,7.200000,0.000000
720575940629327659,DNa02_right,49.033333,6.466667
720575940604737708,DNa02_left,0.866667,17.466667
720575940618238523,MN9_right,0.100000,0.000000


In [ ]:
df_rate.to_csv('./results/df_rate.csv') #Save the data.